# 11f Texas and Wisconsin

Everything so far is California. If the findings are real, they should hold in the other two states. Here we rerun
the pipeline on Texas and Wisconsin with every setting frozen at its California value: the scoping rule, the
detector's burn-in, baseline and thresholds, the PELT penalty, the forecasting models and the stock rules. Only the
stores, the SNAP column and the products change. We wrote down seven conclusions to check before running it:

| # | conclusion | holds if |
|---|---|---|
| R1 | shifts are common | the CUSUM flags 50-80% of products |
| R2 | LightGBM beats seasonal naive | lower median MASE and paired p < 0.05 |
| R3 | LightGBM at least as good as ETS | median MASE not higher than ETS |
| R4 | retraining after a shift does not help | fails the "helps" rule |
| R5 | static LightGBM is cheaper than seasonal naive | lower total cost at 95%, base costs |
| R6 | retraining is not cheaper than static | the 95% interval of the cost difference is not wholly below 0 |
| R7 | 99% is the cheapest of 90, 95 and 99% | base costs, LightGBM static |

In [1]:
import sys, time
sys.path.append('../src')
import numpy as np
import pandas as pd

from break_detection import to_weekly, cusum_detect
from calibration import seasonal_null, cusum_alarm, pelt_changepoints
from metrics import paired_test
from pipeline import (calibrate_h, intermittency, item_mase, make_origins, price_daily_frame, run_backtest,
                      scope_state, simulate_inventory, state_prices, summarise_inventory)

SEED = 42
BURN_IN, REF_LEN, K = 8, 52, 1.0
PRACTICAL = 0.0074
calib = pd.read_csv("../outputs/tables/break_calibration.csv").set_index('method')
H_FULL, PEN = float(calib.loc['CUSUM', 'value']), float(calib.loc['PELT', 'value'])
ORIGINS = make_origins('2016-05-22', 7)
EVAL = ORIGINS[1:]
POL = [('lightgbm', 'static'), ('lightgbm', 'adaptive'), ('seasonal_naive', 'static')]

sales = pd.read_csv("../data/raw/sales_train_evaluation.csv")
calendar = pd.read_csv("../data/raw/calendar.csv", parse_dates=['date'])
prices = pd.read_csv("../data/raw/sell_prices.csv", dtype={'store_id': 'category'})

## Check: the reusable scoping reproduces California, and the frozen thresholds

In [2]:
ca_long, ca_summary, _ = scope_state(sales, calendar, 'CA')
ref = pd.concat([pd.read_csv(f"../data/interim/{c}_candidates.csv", parse_dates=['date'])
                 for c in ['foods', 'hobbies', 'household']])
same_items = set(ca_long['item_id']) == set(ref['item_id'])
same_rows = len(ca_long) == len(ref) and ca_long['sales'].sum() == ref['sales'].sum()
print(f"CA items {ca_long['item_id'].nunique()} (reference {ref['item_id'].nunique()}); identical item set: {same_items}; "
      f"identical rows and total units: {same_rows}")
print(ca_summary.to_string(index=False))
ca_daily = {i: g[['date', 'sales']].reset_index(drop=True) for i, g in ca_long.groupby('item_id')}
H_FC = calibrate_h(ca_daily, ORIGINS[0])
print(f"Frozen thresholds: full-series h = {H_FULL:.3f}, forecasting h = {H_FC:.3f} (CA, data before {ORIGINS[0].date()}), "
      f"PELT penalty = {PEN:.2f}")
del ca_long, ca_daily, ref

CA items 726 (reference 726); identical item set: True; identical rows and total units: True
state  category  total_items  items_with_leading_zeros  ax_ay_candidates  unit_share_ax_ay
   CA     FOODS         1437                       714               337          0.663593
   CA   HOBBIES          565                       303               106          0.635853
   CA HOUSEHOLD         1047                       548               283          0.676403


Frozen thresholds: full-series h = 14.385, forecasting h = 13.830 (CA, data before 2015-11-08), PELT penalty = 31.03


In [3]:
results, store = [], {}
for state in ['TX', 'WI']:
    t0 = time.time()
    long_df, summary, _ = scope_state(sales, calendar, state)
    daily = {i: g[['date', 'sales']].reset_index(drop=True) for i, g in long_df.groupby('item_id')}
    cat = long_df.groupby('item_id')['category'].first()

    # Breaks (full series) and the seasonal-null false-alarm rate of the frozen thresholds
    rng = np.random.default_rng(SEED)
    brk = []
    for i, d in daily.items():
        v = to_weekly(d).to_numpy()
        nulls, kind = seasonal_null(v, BURN_IN, REF_LEN, n_series=20, rng=rng)
        brk.append({'item_id': i, 'category': cat[i], 'null_kind': kind,
                    'cusum_break': cusum_alarm(v, H_FULL, K, BURN_IN, REF_LEN)[0] is not None,
                    'pelt_break': len(pelt_changepoints(v, PEN, BURN_IN, REF_LEN)) > 0,
                    'cusum_far': np.mean([cusum_alarm(s, H_FULL, K, BURN_IN, REF_LEN)[0] is not None for s in nulls]),
                    'pelt_far': np.mean([len(pelt_changepoints(s, PEN, BURN_IN, REF_LEN)) > 0 for s in nulls])})
    brk = pd.DataFrame(brk)
    print(f"{state}: {len(daily)} items, breaks done ({time.time() - t0:.0f}s)")

    # Forecasting backtest with the frozen CA forecasting threshold
    pw = state_prices(prices, calendar, long_df['item_id'].unique(), state)
    fc, windows, scales = run_backtest(long_df[['item_id', 'date', 'sales']], calendar, pw, ORIGINS, H_FC,
                                       snap_col=f'snap_{state}', run_ets=True, log=lambda m: None)
    fc.to_parquet(f"../data/processed/replication_forecasts_{state}.parquet", index=False)
    print(f"{state}: backtest done ({time.time() - t0:.0f}s)")

    # Inventory
    days = pd.date_range(EVAL[0] + pd.Timedelta(days=1), periods=28 * 6)
    kp = simulate_inventory(fc, ORIGINS, price_daily_frame(pw, days), POL, [0.90, 0.95, 0.99], 7,
                            intermittency(long_df, ORIGINS[0]))
    print(f"{state}: inventory done ({time.time() - t0:.0f}s)")
    store[state] = dict(summary=summary, brk=brk, fc=fc, windows=windows, scales=scales, kp=kp, cat=cat)

TX: 529 items, breaks done (138s)


TX: backtest done (1469s)


TX: inventory done (1529s)


WI: 598 items, breaks done (156s)


WI: backtest done (1597s)


WI: inventory done (1663s)


## Scope, shifts and false alarms

In [4]:
scope = pd.concat([s['summary'] for s in store.values()], ignore_index=True)
scope.to_csv("../outputs/tables/replication_scope.csv", index=False)
print(scope.to_string(index=False))
rows = []
for state, s in store.items():
    b = s['brk']
    for label, g in [('ALL', b)] + list(b.groupby('category')):
        rows.append({'state': state, 'category': label, 'n_items': len(g),
                     'cusum_break_rate': g['cusum_break'].mean(), 'pelt_break_rate': g['pelt_break'].mean(),
                     'cusum_far_seasonal_null': g['cusum_far'].mean(), 'pelt_far_seasonal_null': g['pelt_far'].mean()})
brk_tbl = pd.DataFrame(rows)
brk_tbl.to_csv("../outputs/tables/replication_breaks.csv", index=False)
print(brk_tbl.round(4).to_string(index=False))

state  category  total_items  items_with_leading_zeros  ax_ay_candidates  unit_share_ax_ay
   TX     FOODS         1437                       710               228          0.623323
   TX   HOBBIES          565                       306                82          0.390781
   TX HOUSEHOLD         1047                       551               219          0.652014
   WI     FOODS         1437                       739               294          0.615542
   WI   HOBBIES          565                       306                82          0.501861
   WI HOUSEHOLD         1047                       554               222          0.664862
state  category  n_items  cusum_break_rate  pelt_break_rate  cusum_far_seasonal_null  pelt_far_seasonal_null
   TX       ALL      529            0.6635           0.8110                   0.1121                  0.1531
   TX     FOODS      228            0.7412           0.8728                   0.1395                  0.1785
   TX   HOBBIES       82            

## Forecast accuracy, retraining and stock

In [5]:
rng = np.random.default_rng(SEED)
mrows, arows, conclusions = [], [], []
for state, s in store.items():
    im = item_mase(s['fc'], s['scales'], EVAL)
    im['category'] = im['item_id'].map(s['cat'])
    st = im[im['variant'] == 'static'].pivot(index='item_id', columns='model', values='mase')
    for model in ['seasonal_naive', 'ets', 'lightgbm']:
        mrows.append({'state': state, 'model': model, 'n_items': st[model].notna().sum(), 'median_item_mase': st[model].median()})
    lgb_naive = paired_test(st['lightgbm'] - st['seasonal_naive'], seed=SEED)
    lgb_ets = paired_test(st['lightgbm'] - st['ets'], seed=SEED)

    # truncate vs static on break item-origins (same per-item averaging as 09a)
    e = s['fc'][(s['fc']['model'] == 'lightgbm') & s['fc']['origin'].isin(EVAL)].assign(
        ae=lambda x: (x['actual'] - x['forecast']).abs())
    e = e.groupby(['item_id', 'origin', 'variant'], as_index=False)['ae'].mean().merge(s['scales'], on=['item_id', 'origin'])
    e['mase'] = e['ae'] / e['mase_scale']
    p = e.pivot_table(index=['item_id', 'origin'], columns='variant', values='mase').reset_index()
    p = p.merge(s['windows'][['item_id', 'origin', 'has_break']], on=['item_id', 'origin'])
    d = p[p['has_break']].assign(diff=lambda x: x['adaptive'] - x['static']).groupby('item_id')['diff'].mean()
    trunc = paired_test(d, seed=SEED)
    helps = trunc['n'] >= 20 and trunc['p_value'] < 0.05 and trunc['median_diff'] < 0 and abs(trunc['median_diff']) >= PRACTICAL
    arows.append({'state': state, 'test': 'lightgbm - seasonal_naive', **lgb_naive})
    arows.append({'state': state, 'test': 'lightgbm - ets', **lgb_ets})
    arows.append({'state': state, 'test': 'truncate - static (break items)', **trunc})

    # Inventory conclusions
    kp = s['kp']
    summ = summarise_inventory(kp)
    s['inv'] = summ.assign(state=state)
    c95 = summ[summ['service_level'] == 0.95].set_index('policy')['total_cost']
    t95 = kp[kp['service_level'] == 0.95].pivot(index='item_id', columns='policy', values='total_cost')
    dd = (t95['lightgbm_adaptive'] - t95['lightgbm_static']).to_numpy()
    boot = dd[rng.integers(0, len(dd), size=(5000, len(dd)))].sum(axis=1)
    lo, hi = np.quantile(boot, [0.025, 0.975])
    static_costs = summ[summ['policy'] == 'lightgbm_static'].set_index('service_level')['total_cost']
    br = s['brk']['cusum_break'].mean()
    med = {m: st[m].median() for m in ['seasonal_naive', 'ets', 'lightgbm']}
    conclusions += [
        (state, 'R1 breaks are common', f"CUSUM break rate {br:.4f}", 0.5 <= br <= 0.8),
        (state, 'R2 LightGBM beats seasonal naive', f"median MASE {med['lightgbm']:.4f} vs {med['seasonal_naive']:.4f}, "
         f"p = {lgb_naive['p_value']:.2e}", med['lightgbm'] < med['seasonal_naive'] and lgb_naive['p_value'] < 0.05),
        (state, 'R3 LightGBM at least as good as ETS', f"median MASE {med['lightgbm']:.4f} vs ETS {med['ets']:.4f}",
         med['lightgbm'] <= med['ets']),
        (state, 'R4 truncate-adaptation does not help', f"median diff {trunc['median_diff']:+.4f} "
         f"(CI {trunc['ci_low']:+.4f} to {trunc['ci_high']:+.4f}), p = {trunc['p_value']:.4f}, n = {trunc['n']}", not helps),
        (state, 'R5 static LightGBM cheaper than naive (95%)',
         f"{c95['lightgbm_static']:.0f} vs {c95['seasonal_naive_static']:.0f}", c95['lightgbm_static'] < c95['seasonal_naive_static']),
        (state, 'R6 adaptive not cheaper than static (95%)', f"total diff {dd.sum():+.0f}, 95% CI {lo:+.0f} to {hi:+.0f}", not hi < 0),
        (state, 'R7 99% cheapest of 90/95/99', f"cheapest = {static_costs.idxmin():.2f}", static_costs.idxmin() == 0.99),
    ]
pd.DataFrame(mrows).to_csv("../outputs/tables/replication_mase.csv", index=False)
pd.DataFrame(arows).to_csv("../outputs/tables/replication_tests.csv", index=False)
inv_all = pd.concat([s['inv'] for s in store.values()], ignore_index=True)
inv_all.to_csv("../outputs/tables/replication_inventory.csv", index=False)
concl = pd.DataFrame(conclusions, columns=['state', 'conclusion', 'evidence', 'holds'])
concl.to_csv("../outputs/tables/replication_conclusions.csv", index=False)
pd.set_option('display.width', 230)
print(pd.DataFrame(mrows).round(4).to_string(index=False))
print()
print(pd.DataFrame(arows)[['state', 'test', 'n', 'median_diff', 'ci_low', 'ci_high', 'share_improved', 'p_value']].round(5).to_string(index=False))
print()
print(inv_all[['state', 'policy', 'service_level', 'n_items', 'fill_rate', 'cycle_service_level', 'avg_inventory_value', 'total_cost']].round(4).to_string(index=False))
print()
for r in concl.itertuples():
    print(f"{r.state} {'HOLDS' if r.holds else 'DOES NOT HOLD'}: {r.conclusion}  [{r.evidence}]")

state          model  n_items  median_item_mase
   TX seasonal_naive      529            0.9479
   TX            ets      529            0.7393
   TX       lightgbm      529            0.7342
   WI seasonal_naive      598            1.0252
   WI            ets      598            0.8049
   WI       lightgbm      598            0.7786

state                            test   n  median_diff   ci_low  ci_high  share_improved  p_value
   TX       lightgbm - seasonal_naive 529     -0.20544 -0.21340 -0.19602         0.92628  0.00000
   TX                  lightgbm - ets 529     -0.00080 -0.00424  0.00250         0.51040  0.59980
   TX truncate - static (break items) 353      0.00457  0.00223  0.00771         0.41360  0.00002
   WI       lightgbm - seasonal_naive 598     -0.24757 -0.25673 -0.23035         0.92809  0.00000
   WI                  lightgbm - ets 598     -0.01096 -0.01796 -0.00820         0.64381  0.00000
   WI truncate - static (break items) 386      0.00710  0.00519  0.00963   

## What we found

All 14 checks hold. First, the reusable code reproduces California exactly (the same 726 products, rows and units).
Texas has 529 products and Wisconsin 598.

Shifts are about as common: the detector flags 66.4% in Texas and 64.9% in Wisconsin, with false-alarm rates on the
realistic break-free series of 11.2% and 11.7%, close to California's 11.9%. LightGBM beats seasonal naive in both
(0.734 against 0.948 in Texas, 0.779 against 1.025 in Wisconsin). Against ETS it is only just ahead in Texas (0.734
against 0.739, not significant) but clearly ahead in Wisconsin (0.779 against 0.805). Retraining after a shift makes
LightGBM worse in both, by +0.0046 and +0.0071 MASE. Static LightGBM is cheaper than seasonal naive in both, and
retraining costs more in Texas (+$2,234, interval wholly above zero) and is indistinguishable from static in
Wisconsin (+$168). The 99% target is the cheapest in both states.

## What comes next

`12_advanced_models_wip` starts on more advanced models. `90_powerbi_export` and `91_dashboard_export` package the results.

In [6]:
# files this notebook wrote
from pathlib import Path
ROOT = Path('..').resolve()
written = [ROOT / p for p in ['outputs/tables/replication_scope.csv', 'outputs/tables/replication_breaks.csv', 'outputs/tables/replication_mase.csv', 'outputs/tables/replication_tests.csv', 'outputs/tables/replication_inventory.csv', 'outputs/tables/replication_conclusions.csv']]
for p in written:
    print(p.relative_to(ROOT), f"{p.stat().st_size / 1e3:,.0f} KB")

outputs\tables\replication_scope.csv 0 KB
outputs\tables\replication_breaks.csv 1 KB
outputs\tables\replication_mase.csv 0 KB
outputs\tables\replication_tests.csv 1 KB
outputs\tables\replication_inventory.csv 4 KB
outputs\tables\replication_conclusions.csv 1 KB
